# 45분 실습: 먼저 검사할 웨이퍼 100장 고르기
익명 센서값으로 검사 우선순위를 정하고, 같은 수를 골랐을 때 어느 기준이 실제 저수율 웨이퍼를 더 많이 찾는지 비교합니다. 새 노트북이나 중간 CSV·JSON은 필요 없습니다.

**준비:** Colab 왼쪽 파일 패널에서 `연습_3600장.csv`와 `검증_1200장_수율숨김.csv`를 `/content`에 업로드하고 위에서부터 실행하세요. 로컬에서는 `student-kit`의 원본을 찾습니다. 첫 코드 셀은 현재 커널에 데이터 처리용 `pandas`와 그래프용 `matplotlib`를 설치합니다. 이미 있다면 그대로 사용합니다.

In [ ]:
# 현재 커널에 데이터 처리용 pandas와 그래프용 matplotlib를 설치합니다.
%pip install -q pandas matplotlib

## 1. 데이터 확인과 날짜 분할 (0~10분)

### 1.1 연습용·검증용 데이터 미리보기

**실행 전: 왜 두 파일을 먼저 확인하나요?** 두 파일의 입력 항목은 비슷하지만, 정답이 들어 있는 파일과 없는 파일을 구분해야 합니다. `shape`로 행과 열 수를, `head(3)`으로 처음 세 행을 확인합니다. 한 행은 웨이퍼 한 장입니다. 두 표에서 `wafer_id`, 생산 날짜, 장비·레시피, 근무조, 익명 센서 `S01`~`S07`을 찾아보고 수율 열이 어느 쪽에 있는지 비교하세요.

In [ ]:
# CSV를 읽고 표와 그래프를 표시할 도구를 불러옵니다.
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

# Colab 또는 로컬 student-kit에서 원본 CSV 두 개를 찾습니다.
root = Path.cwd()
practice_name = '연습_3600장.csv'
blind_name = '검증_1200장_수율숨김.csv'
folders = (root, root / 'student-kit', root.parent / 'student-kit')
folder = next((path for path in folders if all((path / name).is_file() for name in (practice_name, blind_name))), None)
if folder is None:
    raise FileNotFoundError('학생용 원본 CSV 두 개를 같은 폴더에 업로드하세요.')

# 파일별 크기와 첫 세 행을 나란히 확인합니다.
practice = pd.read_csv(folder / practice_name, encoding='utf-8-sig', dtype={'wafer_id': 'string'})
blind = pd.read_csv(folder / blind_name, encoding='utf-8-sig', dtype={'wafer_id': 'string'})
for label, frame in [('연습용', practice), ('검증용', blind)]:
    row_count, column_count = frame.shape
    print(f'{label}: {row_count:,}장, {column_count}열')
    display(frame.head(3))
print('연습용에만 있는 열:', ', '.join(practice.columns.difference(blind.columns)))

# 수율 정답 열이 연습용에만 있는지 확인합니다.
targets = ['yield_rate', 'yield_status']
if not set(targets).issubset(practice.columns) or set(targets).intersection(blind.columns):
    raise ValueError('수율 열이 있는 연습용과 수율을 가린 검증용을 사용하세요.')

**1.1 실행 후 확인:** 연습용은 3,600장·14열, 검증용은 1,200장·12열입니다. 두 표에 장비·날짜·센서값은 있지만 실제 수율(`yield_rate`)과 저수율 라벨(`yield_status`)은 연습용에만 있습니다. 따라서 연습용은 기준을 만들고 맞았는지 확인하는 데, 검증용은 마지막에 검사할 100장을 고르는 데 사용합니다.

### 1.2 기준 만들기용 데이터 준비

**실행 전: 왜 앞쪽 날짜로 기준을 만드나요?** 센서값이 얼마나 평소와 다른지 알려면 먼저 비교할 평소 값이 필요합니다. 이미 관찰한 날의 데이터로 기준을 만들고, 그보다 늦은 날에 적용하는 실제 사용 상황을 흉내 냅니다. 연습용의 고유 생산 날짜를 정렬한 뒤 앞 70%를 내림해 `fit`으로 고릅니다. 행이 아닌 날짜로 나누므로 같은 날이 양쪽에 섞이지 않습니다. 검증용 CSV는 여기서 사용하지 않습니다.

In [ ]:
# 이후 그래프와 점수 계산에 사용할 센서 및 장비·레시피 열입니다.
sensors = [f'S{number:02d}' for number in range(1, 8)]
keys = ['equipment_id', 'recipe_id']

# 중복 없이 날짜를 정렬해 같은 날이 양쪽에 섞이지 않게 합니다.
practice['production_date'] = pd.to_datetime(practice['production_date'], format='%Y-%m-%d')
dates = practice['production_date'].dropna().drop_duplicates().sort_values().tolist()
if practice['production_date'].isna().any() or len(dates) < 2:
    raise ValueError('빈 생산 날짜가 없어야 하고 서로 다른 날짜가 두 개 이상 필요합니다.')

# 앞 70%의 날짜에 해당하는 행만 기준 만들기에 사용합니다.
fit_date_count = max(1, int(len(dates) * 0.7))
boundary = dates[fit_date_count]
fit = practice.loc[practice['production_date'] < boundary].copy()
print(f'기준 만들기: {len(fit):,}장 / {fit["production_date"].min():%Y-%m-%d} ~ {fit["production_date"].max():%Y-%m-%d}')
print(f'기준 비교 시작일: {boundary:%Y-%m-%d}')

**1.2 실행 후 확인:** 앞쪽은 2,480장이고 마지막 날짜는 2025-10-31입니다. 비교 시작일인 2025-11-01은 여기에 포함되지 않습니다. 이 셀에서 준비한 `fit`으로 다음 단계의 센서 추세를 살펴보고, 이후 장비·레시피별 평소 센서값을 계산합니다. 아직 센서 점수를 계산한 것은 아닙니다.

### 1.3 기준 비교용 데이터와 실제 저수율 수 확인

**실행 전: 왜 뒤쪽 날짜를 따로 보나요?** 기준을 만든 데이터에서 그 기준을 평가하면 실제로 처음 보는 데이터보다 성능이 좋아 보일 수 있습니다. 그래서 기준을 만드는 데 쓰지 않은 뒤쪽 날짜를 `compare`로 따로 둡니다. 여기서는 실제 `LOW`를 세어, 나중에 각 기준이 몇 장을 찾고 몇 장을 놓쳤는지 계산할 기준값을 마련합니다.

In [ ]:
# 기준 만들기에 사용하지 않은 뒤쪽 날짜만 따로 둡니다.
compare = practice.loc[practice['production_date'] >= boundary].copy()

# 실제 저수율 수를 세어 나중에 적중과 누락을 계산할 때 사용합니다.
low_count = compare['yield_status'].eq('LOW').sum()

print(f'기준 비교: {len(compare):,}장 / {boundary:%Y-%m-%d} ~ {compare["production_date"].max():%Y-%m-%d}')
print(f'기준 비교 구간의 실제 저수율: {low_count:,}장')

**실행 후 확인:** 뒤쪽 비교 구간은 1,120장이고, 실제 저수율은 54장입니다. 이 54장은 우리가 고른 장수가 아니라 **비교 구간 안의 전체 정답 수**입니다. 이후 각 기준으로 93장씩 고른 뒤, 그 안에 실제 `LOW`가 몇 장인지 세면 적중, `54 - 적중`은 누락입니다. 예를 들어 24장을 찾았다면 30장을 놓친 것입니다. 이 라벨은 결과를 비교할 때만 사용하고, 수율을 가린 최종 검증용 1,200장의 점수에는 쓰지 않습니다.

## 2. 센서 추세와 저수율 그룹 비교 (10~20분)

### 2.1 살펴볼 센서와 레시피 고르기

**실행 전: 왜 앞쪽 데이터부터 보나요?** 1.2에서 만든 `fit`으로 어떤 센서를 검사 기준에 쓸지 살펴봅니다. 1.3의 뒤쪽 데이터는 3단계에서 기준을 비교할 때까지 센서 선택에 사용하지 않습니다. 먼저 기본값 `S02`로 실행하세요. 다른 센서를 보고 싶다면 `sensor`를 바꾸고, 같은 레시피만 보고 싶다면 `recipe`를 예를 들어 `'R1'`로 바꾼 뒤 2.1~2.3을 다시 실행하세요.

In [ ]:
# sensor를 바꿔 보고, recipe가 None이면 모든 레시피를 봅니다.
sensor = 'S02'
recipe = None
if sensor not in sensors:
    raise ValueError('S01~S07 중 하나를 고르세요.')
view = fit if recipe is None else fit.loc[fit['recipe_id'].eq(recipe)]
if view.empty:
    raise ValueError('선택한 레시피에 데이터가 없습니다.')

print(f'센서: {sensor} / 레시피: {recipe or "전체"} / 웨이퍼: {len(view):,}장')
print(f'센서값이 비어 있는 웨이퍼: {view[sensor].isna().sum():,}장')

**2.1 실행 후 확인:** 기본값 `S02`, 레시피 전체에서는 앞쪽 2,480장을 보고, 센서값이 빈 행은 0장입니다. 설정을 바꾸면 이 숫자도 달라질 수 있습니다.

### 2.2 장비별 일별 중앙값 그래프 읽기

**실행 전: 왜 일별 중앙값을 그리나요?** 하루치 측정값을 대표값 하나로 모으면 장비별 변화 방향을 보기 쉽습니다. 중앙값은 값을 작은 순서로 놓았을 때 가운데에 오는 값입니다.

- **가로축:** 생산 날짜. 같은 날 안의 처리 순서를 뜻하지 않습니다.
- **세로축:** 선택한 센서의 일별 중앙값. 수율이 아닙니다.
- **색/선:** 장비 한 대. **점 하나:** 그 날짜·장비에서 관측된 센서값들의 중앙값이며 웨이퍼 한 장이 아닙니다.

In [ ]:
# 날짜·장비별 센서 중앙값을 구해 장비마다 선 하나를 그립니다.
daily = view.groupby(['production_date', 'equipment_id'])[sensor].median().unstack('equipment_id')
axis = daily.plot(marker='.', figsize=(9, 3.5), title=f'{sensor} daily median | recipe={recipe or "ALL"}')
axis.set(xlabel='Production date', ylabel=f'{sensor} daily median')
axis.legend(title='Equipment')
plt.tight_layout()
plt.show()

**2.2 실행 후: 같은 색의 선 하나를 따라가세요.** 기본 `S02`·전체 레시피에서 ETCH-D는 10월 1일 102.705, 10월 31일 99.025로 마지막 날 중앙값이 첫날보다 약 3.68 낮습니다. 중간 점도 보면서 계속 낮아졌는지, 특정 날짜만 달랐는지 구분하세요. 이 그래프에는 수율이 없고, 같은 날 안의 처리 순서도 알 수 없습니다.

### 2.3 같은 장비에서 정상·저수율 비교

**실행 전: 왜 표를 따로 보나요?** 추세 그래프만으로는 값의 변화가 저수율과 함께 나타났는지 알 수 없습니다. 앞쪽 데이터의 정답 라벨로 같은 장비 안에서 `NORMAL`과 `LOW`를 나눕니다. `count`는 센서값이 기록된 웨이퍼 수, `median`은 그 센서값의 중앙값입니다. 두 그룹의 중앙값을 비교하되, 관측 수가 적거나 빠진 그룹은 해석에 주의하세요.

In [ ]:
# 같은 장비 안에서 정상·저수율 그룹의 관측 수와 중앙값을 비교합니다.
group_summary = view.groupby(['equipment_id', 'yield_status'])[sensor].agg(['count', 'median'])
display(group_summary)

**2.3 실행 후: 두 결과를 연결해 보세요.** 기본 `S02`·전체 레시피에서 ETCH-D는 저수율 그룹 13장의 중앙값이 99.89, 정상 그룹 607장의 중앙값이 102.62입니다. 그래프에서는 이 장비의 10월 말 값이 10월 초보다 낮았고, 표에서는 저수율 그룹의 중앙값이 더 낮았습니다. **두 관찰은 같은 날짜의 웨이퍼에서 일어난 일이라는 뜻도, S02가 저수율의 원인이라는 뜻도 아닙니다.** 표에는 날짜가 없고 두 그룹의 장수도 다릅니다. 필요하면 같은 레시피로 2.1~2.3을 다시 실행해 비교하세요.

**학생 기록 한 문장:** “`[장비]`의 `[센서]`는 `[기간과 변화]`였고, `[정상·저수율 비교]`가 관찰됐다.” 이 관찰을 3단계의 센서 후보로 쓰고, 실제 선별에 도움이 되는지는 뒤쪽 날짜에서 적중 수로 확인합니다.

## 3. A·B·무작위의 적중 비교 (20~35분)

### 3.1 센서 조합과 점수 계산 준비

**실행 전: 왜 두 가지 기준을 만드나요?** 2단계에서 살펴본 센서 하나만 써도 충분한지, 여러 센서를 함께 쓰면 더 잘 찾는지 비교합니다. A는 2.1의 `sensor` 하나, B는 `rules`에 적은 서로 다른 센서 2~3개를 씁니다. 기본 조합은 실습 예시입니다.

장비·레시피별 평소 중앙값을 **앞쪽 `fit`에서만** 구합니다. 평소 값과의 차이가 클수록 점수가 높습니다. 센서마다 값의 범위가 다르므로 차이를 센서별 표준편차로 나눕니다. 예를 들어 차이가 2이고 표준편차가 4라면 점수는 0.5입니다. B는 여러 센서 점수의 평균입니다. 장비·레시피 조합의 관측이 5개 미만이면 전체 중앙값을 쓰고, 센서값이 비면 해당 센서 점수를 0으로 둡니다. 수율은 점수에 사용하지 않습니다.

In [ ]:
import math

# 기준 A는 센서 하나, 기준 B는 서로 다른 센서 2~3개를 사용합니다.
rules = {'A': [sensor], 'B': ['S01', 'S02', 'S03']}
if len(rules['A']) != 1 or len(rules['B']) not in (2, 3) or len(set(rules['B'])) != len(rules['B']):
    raise ValueError('A는 1개, B는 서로 다른 2~3개 센서를 고르세요.')
chosen_sensors = sorted(set(rules['A'] + rules['B']))
if any(column not in sensors for column in chosen_sensors):
    raise ValueError('S01~S07에서 고르세요.')

# 앞쪽 데이터에서 장비·레시피별 평소 값과 센서별 표준편차를 구합니다.
stats = {}
for column in chosen_sensors:
    center = fit[column].median()
    scale = fit[column].std(ddof=0)
    if not math.isfinite(scale) or scale <= 0:
        raise ValueError(f'{column}: 값의 변화가 있는 다른 센서를 고르세요.')
    groups = fit.groupby(keys)[column].agg(['count', 'median'])
    group_centers = groups.loc[groups['count'] >= 5, 'median'].to_dict()
    stats[column] = (center, scale, group_centers)


# 평소 값과의 차이를 표준편차로 나눈 뒤 센서별 점수를 평균냅니다.
def score_rows(frame, columns):
    equipment_recipes = list(frame[keys].itertuples(index=False, name=None))
    deviations = {}
    for column in columns:
        center, scale, group_centers = stats[column]
        expected = [group_centers.get(pair, center) for pair in equipment_recipes]
        deviations[column] = frame[column].sub(expected).abs().div(scale).fillna(0.0)
    return pd.DataFrame(deviations).mean(axis=1)


for rule, columns in rules.items():
    print(f'기준 {rule}: {", ".join(columns)}')

**3.1 실행 후:** A와 B에 어떤 센서가 들어갔는지 확인하세요. 아직 웨이퍼를 고른 것은 아닙니다. 방금 준비한 `score_rows()`는 뒤쪽 비교 구간과 최종 검증용 모두에 같은 방식으로 점수를 매깁니다.

### 3.2 같은 수의 웨이퍼 고르기

**실행 전: 왜 모두 같은 장수를 고르나요?** 더 많이 고르면 적중 수도 늘 수 있으므로 검사 예산을 같게 맞춥니다. 실전의 검사 비율인 **1,200장 중 100장**을 비교 구간 1,120장에 적용해 93장으로 반올림합니다. A와 B는 높은 점수순으로, 무작위는 재현 가능한 고정 시드로 고릅니다. 이때 실제 수율은 사용하지 않습니다.

In [ ]:
# 비교 구간에서 A·B·무작위가 같은 장수를 고릅니다.
if len(blind) < 100:
    raise ValueError('검증용 파일에 웨이퍼가 100장 이상 있어야 합니다.')
budget = max(1, round(len(compare) * 100 / len(blind)))
comparison_inputs = compare.drop(columns=targets)
selections = {}
for rule, columns in rules.items():
    score = score_rows(comparison_inputs, columns)
    selections[rule] = score.sort_values(ascending=False, kind='stable').index[:budget]
selections['무작위'] = compare.sample(n=budget, random_state=42).index

for rule, indices in selections.items():
    print(f'{rule}: {len(indices):,}장 선택')

**3.2 실행 후:** 현재 데이터에서는 세 기준 모두 93장을 골랐는지 확인하세요. 아직 어느 쪽이 더 많은 저수율을 찾았는지는 모릅니다.

### 3.3 실제 저수율 라벨로 결과 비교하기

**실행 전: 표에서 무엇을 보나요?** 선택을 끝낸 다음에 정답인 `LOW`를 확인합니다. **적중**은 고른 것 중 실제 저수율 수, **누락**은 전체 저수율에서 적중을 뺀 수, **선택 중 저수율 비율**은 적중을 고른 장수로 나눈 값입니다. 고른 장수가 같으므로 적중 수를 바로 비교할 수 있습니다.

In [ ]:
# 선택이 끝난 뒤 실제 저수율 라벨로 적중과 누락을 셉니다.
low = compare['yield_status'].eq('LOW')
records = []
for rule, indices in selections.items():
    hits = int(low.loc[indices].sum())
    records.append({
        '기준': rule,
        '센서': ', '.join(rules.get(rule, [])) or '-',
        '선택': len(indices),
        '적중': hits,
        '누락': int(low.sum()) - hits,
        '선택 중 저수율 (%)': round(100 * hits / len(indices), 1),
    })
summary = pd.DataFrame(records).set_index('기준')
display(summary)

**3.3 실행 후:** 기본 예시에서 A·B·무작위는 각각 93장을 골랐고, 적중은 24장·13장·9장입니다. A는 실제 저수율 54장 중 24장을 찾아 30장을 놓쳤고, 고른 93장 중 25.8%가 저수율이었습니다. 센서를 바꾸면 결과도 달라집니다. 같은 뒤쪽 데이터의 결과를 보면서 센서 조합을 반복해서 바꾸면 성능이 실제보다 좋아 보일 수 있습니다.

## 4. 기준 확정과 상위 100장 저장 (35~45분)

### 4.1 실전에 적용할 기준 하나 정하기

**실행 전: 왜 먼저 기준을 확정하나요?** 검증용은 실제 수율이 숨겨져 있어 현장에서 어떤 기준이 더 맞는지 비교할 수 없습니다. 3.3의 적중 결과를 보고 `final_rule`에 `'A'` 또는 `'B'`를 적으세요. `''`로 두면 적중 수가 큰 기준을, 동률이면 더 단순한 A를 사용합니다. 무작위는 비교용이고 실전에 선택하는 기준은 A 또는 B입니다.

In [ ]:
# 기준을 비워 두면 적중 수가 큰 쪽을 선택합니다. 동률이면 A입니다.
final_rule = ''
recommended = 'A' if summary.loc['A', '적중'] >= summary.loc['B', '적중'] else 'B'
selected_rule = final_rule or recommended
if selected_rule not in rules:
    raise ValueError('A 또는 B를 입력하세요.')
print(f'선택 기준: {selected_rule} ({", ".join(rules[selected_rule])}), 비교 적중 {summary.loc[selected_rule, "적중"]}장')

**4.1 실행 후:** 선택한 기준·센서·비교 적중 수를 확인하고 선택 이유를 한 문장으로 남기세요. 예: “같은 93장을 골랐을 때 A가 저수율을 더 많이 찾아 A를 골랐다.” 무작위보다 못했다면 그 사실도 기록하세요.

### 4.2 검증용에서 우선 검사할 100장 고르기

**실행 전: 왜 그대로 적용하나요?** 검증용에는 정답이 없으므로 앞쪽 데이터로 만든 계산 방식과 4.1에서 확정한 기준을 그대로 씁니다. 점수가 높은 순으로 100장을 고르고, 점수가 같으면 원본 입력 순서를 유지합니다. 이것은 동점 처리 방식이지 웨이퍼 처리 순서를 가정하는 것이 아닙니다.

In [ ]:
# 확정한 기준을 검증용에 적용해 점수순으로 상위 100개 ID를 뽑습니다.
ranking = score_rows(blind, rules[selected_rule])
picked = ranking.sort_values(ascending=False, kind='stable').index[:100]
submission = blind.loc[picked, ['wafer_id']]
ids = submission['wafer_id']
if len(ids) != 100 or not ids.is_unique or ids.fillna('').str.strip().eq('').any():
    raise ValueError('중복이나 빈 값이 없는 웨이퍼 ID 100개가 필요합니다.')
print(f'검사 후보: {len(submission):,}장')
display(submission.head().reset_index(drop=True))

**4.2 실행 후:** `100장`이 표시되는지 확인하고 첫 다섯 ID를 살펴보세요. 실제로 저수율인지 판단하려면 나중에 수율이 공개되어야 합니다. 이 시점에는 아직 파일을 저장하지 않았습니다.

### 4.3 제출용 CSV 저장하고 검증하기

**실행 전: 왜 저장한 파일을 다시 읽나요?** 제출할 때는 화면의 표가 아니라 실제 CSV를 사용합니다. 다시 읽은 파일이 방금 고른 ID 목록과 동일한지 확인하면 열이나 행이 달라지는 실수를 잡을 수 있습니다.

In [ ]:
# CSV를 저장한 뒤 다시 읽어 원래 제출 목록과 같은지 확인합니다.
output = root / 'outputs' / 'results' / '검증_우선검사목록.csv'
output.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output, index=False, encoding='utf-8-sig')
saved = pd.read_csv(output, encoding='utf-8-sig', dtype={'wafer_id': 'string'})
if not saved.equals(submission.reset_index(drop=True)):
    raise ValueError('저장한 CSV의 열과 웨이퍼 ID를 확인하세요.')
print(f'저장 및 검증 완료: {output} (100장)')

**4.3 실행 후:** 저장 경로와 `(100장)` 메시지를 확인하세요. 파일에는 `wafer_id` 한 열만 있고 헤더를 제외한 데이터 행은 100개입니다. 로컬에서는 프로젝트의 `outputs/results/검증_우선검사목록.csv`를 확인하세요. Colab 기본 환경에서는 파일 패널의 `/content/outputs/results/`에서 내려받을 수 있습니다. 이것은 검사 목록이지 적중 결과표가 아닙니다.

## 오전 실습을 마치며
그래프 1개, 비교표 1개, 기준 선택 이유 1문장을 남기세요. 오후에는 같은 비교 구간·검사 수를 고정하고, 여러 센서를 쓰는 기준에서 하나씩 제외해 적중 수 변화를 기록합니다. 이 순위는 **해당 판별 기준에서의 기여도**이지 실제 공정의 원인 순위가 아닙니다. 센서명이 19:00에 공개되면 공정과 연결해 원인 가설을 작성하세요. 실제 수율 공개 후에는 최초 100장의 적중·누락부터 기록합니다.

학생용 기본 파이썬 코드는 오후 과제용 출발점입니다. 그대로 실행하면 TODO 안내에서 멈춥니다. TODO 1·2를 구현하면 연습용 전체에서 상위 300장을 평가하고 검증용 상위 100장을 저장합니다. 이 노트북의 날짜 분할 비교와는 별도입니다.